Back Testing

In [198]:
from datetime import date, datetime
import pandas as pd
import yfinance as yf
import time
from Alert import Alert
pd.options.mode.chained_assignment = None  # default='warn'
alert = Alert('1h','GGAL')

In [ ]:
input:
    
     Titulo  : example: GGAL
     frequencia de tick : 1h (frequencia mais alta)
     dftitulo : vista do BD sqtitulosalpha.bd (testar ultimos periodas)
Parametros a variar para back testing ( definir rangos de variação)
     k, d, smooth (parametros do stch)
     dayM  (frequencia media) (multiplicador da frequencia mais alta) exemplo: 5
     semM  (frequencia baixa) (multiplicador da frequencia media) exemplo: 7

In [137]:
dataini = '2020-04-03 19:30:00'
datafim = '2023-04-03 19:30:00'

In [156]:
import sqlite3
import pandas as pd

# Caminho para o banco de dados
caminho_bd = r'C:\Users\scitr\anaconda_projects\Trading_System\Dados_Fontes\Alpha_Vantage\sqtitulosalpha.db'

# Conectando ao banco
conexao = sqlite3.connect(caminho_bd)

# Lendo a view
#consulta = 'SELECT * FROM vwtitulosdados ORDER BY datetime'
consulta = f"""
SELECT * FROM vwtitulosdados
WHERE datetime BETWEEN '{dataini}' AND '{datafim}'
ORDER BY datetime
"""

dftitulosdados = pd.read_sql_query(consulta, conexao)

# Fechando a conexão
conexao.close()

# Exibindo os primeiros registros para conferir
display(dftitulosdados)

,symbol,moeda,intervalo,datetime,open,high,low,close,volume
0,GGAL,USD,60min,2020-04-06 08:00:00,5.8034,5.8034,5.8034,5.8034,100.0
1,GGAL,USD,60min,2020-04-06 09:00:00,6.0432,6.2510,5.9952,6.0832,114582.0
2,GGAL,USD,60min,2020-04-06 10:00:00,6.0945,6.0945,5.6635,5.7954,159266.0
3,GGAL,USD,60min,2020-04-06 11:00:00,5.7954,5.8034,5.4837,5.6835,242263.0
4,GGAL,USD,60min,2020-04-06 12:00:00,5.6715,5.8354,5.6355,5.8274,186504.0
...,...,...,...,...,...,...,...,...,...
6578,GGAL,USD,60min,2023-04-03 12:00:00,9.5499,9.6357,9.5499,9.5756,29258.0
6579,GGAL,USD,60min,2023-04-03 13:00:00,9.5671,9.6872,9.5671,9.6700,48527.0
6580,GGAL,USD,60min,2023-04-03 14:00:00,9.6785,9.7447,9.6529,9.7301,32877.0
6581,GGAL,USD,60min,2023-04-03 15:00:00,9.7386,9.7816,9.6529,9.7558,143309.0


In [158]:

i = '1h'
K = 16  
D = 5    
smoth = 5  
dayM = 8  
semM = 4


In [160]:
# Stochastic calculation
def stochastic(df, i, K, D, smoth):
        
    df["k"] = (100. * (df.close - df.low.rolling(K).min()) /
        (df.high.rolling(K).max() - df.low.rolling(K).min()))
    
    df["k" + i] = df.k.rolling(smoth).mean()
    df["d" + i] = df["k" + i].rolling(D).mean()
    
    df.drop(columns=["k"], inplace=True)  

    return df
dfstoch = stochastic(dftitulosdados, i, K, D, smoth)
display (dfstoch)

,symbol,moeda,intervalo,datetime,open,high,low,close,volume,k1h,d1h
0,GGAL,USD,60min,2020-04-06 08:00:00,5.8034,5.8034,5.8034,5.8034,100.0,NaN,NaN
1,GGAL,USD,60min,2020-04-06 09:00:00,6.0432,6.2510,5.9952,6.0832,114582.0,NaN,NaN
2,GGAL,USD,60min,2020-04-06 10:00:00,6.0945,6.0945,5.6635,5.7954,159266.0,NaN,NaN
3,GGAL,USD,60min,2020-04-06 11:00:00,5.7954,5.8034,5.4837,5.6835,242263.0,NaN,NaN
4,GGAL,USD,60min,2020-04-06 12:00:00,5.6715,5.8354,5.6355,5.8274,186504.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
6578,GGAL,USD,60min,2023-04-03 12:00:00,9.5499,9.6357,9.5499,9.5756,29258.0,35.345039,24.486408
6579,GGAL,USD,60min,2023-04-03 13:00:00,9.5671,9.6872,9.5671,9.6700,48527.0,43.814248,30.270650
6580,GGAL,USD,60min,2023-04-03 14:00:00,9.6785,9.7447,9.6529,9.7301,32877.0,53.958531,37.149336
6581,GGAL,USD,60min,2023-04-03 15:00:00,9.7386,9.7816,9.6529,9.7558,143309.0,64.423364,45.395125


In [165]:
def set_test( df, intervalo, k, d, smth, dayM, semM):
        
        # Convert time
        #df["time"] = pd.to_datetime(df.index, utc=True)
        #df['timeArg'] = df['time'].dt.tz_convert('America/Argentina/Buenos_Aires')        
        #df['time'] = df['time'].dt.tz_convert(None)

        df = stochastic(df, "hora", k, d, smth)
        df = stochastic(df, "dia", k*dayM, d*dayM, smth*dayM)
        df = stochastic(df, "sem", k*dayM*semM, d*dayM*semM, smth*dayM*semM)
    
        return df
dfstochhlc= set_test(dfstoch, i , K, D, smoth, dayM , semM )
display(dfstochhlc )
%time set_test(dfstoch, i , K, D, smoth, dayM , semM ) 

,symbol,moeda,intervalo,datetime,open,high,low,close,volume,k1h,d1h,khora,dhora,kdia,ddia,ksem,dsem
0,GGAL,USD,60min,2020-04-06 08:00:00,5.8034,5.8034,5.8034,5.8034,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,GGAL,USD,60min,2020-04-06 09:00:00,6.0432,6.2510,5.9952,6.0832,114582.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,GGAL,USD,60min,2020-04-06 10:00:00,6.0945,6.0945,5.6635,5.7954,159266.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,GGAL,USD,60min,2020-04-06 11:00:00,5.7954,5.8034,5.4837,5.6835,242263.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,GGAL,USD,60min,2020-04-06 12:00:00,5.6715,5.8354,5.6355,5.8274,186504.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6578,GGAL,USD,60min,2023-04-03 12:00:00,9.5499,9.6357,9.5499,9.5756,29258.0,35.345039,24.486408,35.345039,24.486408,45.055766,33.328487,48.012866,70.547223
6579,GGAL,USD,60min,2023-04-03 13:00:00,9.5671,9.6872,9.5671,9.6700,48527.0,43.814248,30.270650,43.814248,30.270650,45.854905,33.918139,47.710336,70.311149
6580,GGAL,USD,60min,2023-04-03 14:00:00,9.6785,9.7447,9.6529,9.7301,32877.0,53.958531,37.149336,53.958531,37.149336,46.699935,34.512110,47.428408,70.073438
6581,GGAL,USD,60min,2023-04-03 15:00:00,9.7386,9.7816,9.6529,9.7558,143309.0,64.423364,45.395125,64.423364,45.395125,47.601915,35.110265,47.142843,69.834096


CPU times: total: 31.2 ms
Wall time: 34.3 ms


,symbol,moeda,intervalo,datetime,open,high,low,close,volume,k1h,d1h,khora,dhora,kdia,ddia,ksem,dsem
0,GGAL,USD,60min,2020-04-06 08:00:00,5.8034,5.8034,5.8034,5.8034,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,GGAL,USD,60min,2020-04-06 09:00:00,6.0432,6.2510,5.9952,6.0832,114582.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,GGAL,USD,60min,2020-04-06 10:00:00,6.0945,6.0945,5.6635,5.7954,159266.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,GGAL,USD,60min,2020-04-06 11:00:00,5.7954,5.8034,5.4837,5.6835,242263.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,GGAL,USD,60min,2020-04-06 12:00:00,5.6715,5.8354,5.6355,5.8274,186504.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6578,GGAL,USD,60min,2023-04-03 12:00:00,9.5499,9.6357,9.5499,9.5756,29258.0,35.345039,24.486408,35.345039,24.486408,45.055766,33.328487,48.012866,70.547223
6579,GGAL,USD,60min,2023-04-03 13:00:00,9.5671,9.6872,9.5671,9.6700,48527.0,43.814248,30.270650,43.814248,30.270650,45.854905,33.918139,47.710336,70.311149
6580,GGAL,USD,60min,2023-04-03 14:00:00,9.6785,9.7447,9.6529,9.7301,32877.0,53.958531,37.149336,53.958531,37.149336,46.699935,34.512110,47.428408,70.073438
6581,GGAL,USD,60min,2023-04-03 15:00:00,9.7386,9.7816,9.6529,9.7558,143309.0,64.423364,45.395125,64.423364,45.395125,47.601915,35.110265,47.142843,69.834096


In [168]:
#insere  'COMPRA' no campo oper do df no registro i 
def set_compra( df, i):
        df.loc[i, "oper"] = "COMPRA"
    
#insere  'VENTA' no campo oper do df no registro i 
def set_venta( df, i):
        df.loc[i, "oper"] = "VENTA"


def close_long( df, i, compra, venta):
        df.loc[i, "oper"] = "CLOSELONG"
        if compra !=0.:
            df.loc[i, "long"] = (venta - compra) / compra


def close_short( df, i, compra, venta):
        df.loc[i, "oper"] = "CLOSESHORT"
        if venta !=0.:
            df.loc[i, "short"] = (venta - compra) / venta

In [170]:
def set_signals(df):
        state = ""
        df["compra"] = False
        df["venta"] = False
        df["closeLong"] = False
        df["closeShort"] = False
        df["entradaLongH"] = False
        df["entradaShortH"] = False
        df["oper"] = ""
        df["long"] = ""
        df["short"] = ""
        df["state"] = "neutro"

        compra = 0.
        venta = 0.
#alert.py part 2
        
# outros intervalos
        lastEntradaLongH = False
        lastEntradaShortH = False
            
        for i in df.index:      
            l = df.loc[i]
            df.loc[i, "state"] = state

        #for i, l in df.iterrows():
            #df.at[i, "state"] = state
            khora = float(l.khora)
            dhora = float(l.dhora)
            kdia = float(l.kdia)
            ddia = float(l.ddia)
            ksem = float(l.ksem)
            dsem = float(l.dsem)
            
            if (khora > 20) and (khora > dhora):                   # "si stoch high EM CONDIÇÃO DE COMPRA          
                    df.at[i, "entradaLongH"] = True     
        
                # ESTRATEGIA 1 - Stochastico alineado y esperar cruce
                #if (l.khora > 20. and l.khora > l.dhora):
                    #df.loc[i, "entradaLongH"] = True
                    if (not lastEntradaLongH):                       # si  la entradalong anterior es true  
                        if (kdia > 20. and kdia > ddia):                 # si stch midfre em condição de compra
                            if(state == "venta"):                           # si estabamos en short 
                                df.loc[i, "closeShort"] = True                 # fechamos short
                                state = "neutro"                               # mudamos estado a neutro
                                compra = l.close                               
                                close_short(df, i, compra, venta)              # calcula ganancia porcentual y coloca na coluna short
                            if (ksem > 80. or ksem > dsem):                  #stch lowfre acima media d ou acima de 80 , para evitar saida anticipada
                                df.loc[i, "compra"] = True                       # compramos long                  
                                if state != "compra":                             # si no tenihamos comprado
                                    state = "compra"                                 # mudamos estado a compra
                                    compra = l.close                                 
                                    set_compra(df, i)                                # mudamos oper a COMPRA
            else:
                df.loc[i, "entradaLongH"] = False                   # si no esta  em condição de compra 

            if (l.khora < 80. and l.khora < l.dhora):
                    df.loc[i, "entradaShortH"] = True
                    if (not lastEntradaShortH):
                        if(l.kdia < 80. and l.kdia < l.ddia):
                            if(state == "compra"):
                                df.loc[i, "closeLong"] = True
                                state = "neutro"
                                venta = l.close
                                close_long(df, i, compra, venta)
                            if(l.ksem < 20. or l.ksem < l.dsem):
                                df.loc[i, "venta"] = True
                                if state != "venta":
                                    state = "venta"
                                    venta = l.close
                                    set_venta(df, i)
            else:
                    df.loc[i, "entradaShortH"] = False

            lastEntradaLongH = df.loc[i, "entradaLongH"]
            lastEntradaShortH = df.loc[i, "entradaShortH"]
                    
        
        return df
    
dfsignal= set_signals(dfstochhlc)          
#display(dfsignal)

In [194]:
#df_filtrado = dfsignal[dfsignal['compra'].notna() | dfsignal['venta'].notna()]
#df_filtrado = dfsignal[dfsignal['state'].notna()]
df_filtrado = dfsignal[(dfsignal['state'].notna()) & (dfsignal['state'] == 'venta')]
#df_filtrado = dfsignal[dfsignal['compra'] == True]
df_filtrado = dfsignal[dfsignal['oper'] != '']

pd.set_option('display.max_rows', None)  # Mostra todas as linhas

print(f"Total de registros com 'state' preenchido: {len(df_filtrado)}")
display(df_filtrado.iloc[:, 11:])



Total de registros com 'state' preenchido: 87


,khora,dhora,kdia,ddia,ksem,dsem,compra,venta,closeLong,closeShort,entradaLongH,entradaShortH,oper,long,short,state
719,58.805532,54.782934,77.560404,69.456308,84.667712,NaN,True,False,False,False,True,False,COMPRA,,,
767,56.570574,66.085405,66.369308,78.457399,86.460965,NaN,False,False,True,False,False,True,CLOSELONG,-0.069267,,compra
846,6.715225,6.755678,30.931970,48.482412,79.710350,84.534459,False,True,False,False,False,True,VENTA,,,neutro
970,43.816521,43.814087,21.067897,12.366351,33.094373,62.534045,False,False,False,True,True,False,CLOSESHORT,,0.081937,venta
1060,51.218447,51.755551,24.412361,40.063877,15.071505,32.399684,False,True,False,False,False,True,VENTA,,,neutro
1195,27.500680,17.702370,20.897295,17.598614,4.512878,10.439022,False,False,False,True,True,False,CLOSESHORT,,0.199556,venta
1258,13.224721,14.984595,25.773712,30.182975,4.880179,6.585868,False,True,False,False,False,True,VENTA,,,neutro
1299,27.344972,15.490774,28.014453,23.058807,4.739077,5.355476,False,False,False,True,True,False,CLOSESHORT,,-0.029422,venta
1325,59.038095,65.853663,25.329733,27.756764,4.365731,4.901127,False,True,False,False,False,True,VENTA,,,neutro
1368,86.525592,86.279734,34.938730,25.590686,4.898888,4.620799,True,False,False,True,True,False,COMPRA,,-0.048343,venta


In [171]:
%time set_signals(dfstochhlc)

CPU times: total: 11 s
Wall time: 11.3 s


,symbol,moeda,intervalo,datetime,open,high,low,close,volume,k1h,...,compra,venta,closeLong,closeShort,entradaLongH,entradaShortH,oper,long,short,state
0,GGAL,USD,60min,2020-04-06 08:00:00,5.8034,5.8034,5.8034,5.8034,100.0,NaN,...,False,False,False,False,False,False,,,,
1,GGAL,USD,60min,2020-04-06 09:00:00,6.0432,6.2510,5.9952,6.0832,114582.0,NaN,...,False,False,False,False,False,False,,,,
2,GGAL,USD,60min,2020-04-06 10:00:00,6.0945,6.0945,5.6635,5.7954,159266.0,NaN,...,False,False,False,False,False,False,,,,
3,GGAL,USD,60min,2020-04-06 11:00:00,5.7954,5.8034,5.4837,5.6835,242263.0,NaN,...,False,False,False,False,False,False,,,,
4,GGAL,USD,60min,2020-04-06 12:00:00,5.6715,5.8354,5.6355,5.8274,186504.0,NaN,...,False,False,False,False,False,False,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6578,GGAL,USD,60min,2023-04-03 12:00:00,9.5499,9.6357,9.5499,9.5756,29258.0,35.345039,...,False,False,False,False,True,False,,,,neutro
6579,GGAL,USD,60min,2023-04-03 13:00:00,9.5671,9.6872,9.5671,9.6700,48527.0,43.814248,...,False,False,False,False,True,False,,,,neutro
6580,GGAL,USD,60min,2023-04-03 14:00:00,9.6785,9.7447,9.6529,9.7301,32877.0,53.958531,...,False,False,False,False,True,False,,,,neutro
6581,GGAL,USD,60min,2023-04-03 15:00:00,9.7386,9.7816,9.6529,9.7558,143309.0,64.423364,...,False,False,False,False,True,False,,,,neutro
